# Kaggle Pipeline: YOLO Then U-Net
This notebook builds the dataset generator, creates the MNISTDD-RGB dataset, then trains YOLO detection first and U-Net segmentation second (sequentially).
When available, all GPUs are used for each training stage.

In [1]:
!nvidia-smi || true
!python3 --version

Wed Apr  1 13:30:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!apt-get update -qq
!apt-get install -y -qq cmake ninja-build libopencv-dev
!python3 -m pip install -q ultralytics

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libatspi2.0-0:amd64.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../00-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../01-libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../02-session-migration_0.3.6_amd64.deb ...
Unpacking session-migration (0.3.6) ...
Selecting previously unselected package gsettings-desktop-schemas.
Preparing to unpack .../03-gsettings-desktop-schemas_42.0-1ubuntu1_all.deb ...
Unpacking gsettings-desktop-schemas (42.0-1ubu

In [3]:
%cd /kaggle/working
!rm -rf Minor_Project
!git clone https://github.com/Yash4616/Minor_Project.git
%cd /kaggle/working/Minor_Project

/kaggle/working
Cloning into 'Minor_Project'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 30 (delta 5), reused 29 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 37.48 KiB | 9.37 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/kaggle/working/Minor_Project


In [4]:
%cd /kaggle/working/Temp
!git pull

[Errno 2] No such file or directory: '/kaggle/working/Temp'
/kaggle/working/Minor_Project
Already up to date.


In [5]:
!python3 scripts/download_mnist.py --out_dir /kaggle/working/data

[download] https://ossci-datasets.s3.amazonaws.com/mnist/train-images-idx3-ubyte.gz
[extract] train-images-idx3-ubyte.gz -> train-images-idx3-ubyte
[download] https://ossci-datasets.s3.amazonaws.com/mnist/train-labels-idx1-ubyte.gz
[extract] train-labels-idx1-ubyte.gz -> train-labels-idx1-ubyte
[download] https://ossci-datasets.s3.amazonaws.com/mnist/t10k-images-idx3-ubyte.gz
[extract] t10k-images-idx3-ubyte.gz -> t10k-images-idx3-ubyte
[download] https://ossci-datasets.s3.amazonaws.com/mnist/t10k-labels-idx1-ubyte.gz
[extract] t10k-labels-idx1-ubyte.gz -> t10k-labels-idx1-ubyte
MNIST raw IDX files are ready.


In [6]:
import torch
torch_prefix = torch.utils.cmake_prefix_path
print('Torch CMake prefix:', torch_prefix)
!rm -rf build
!cmake -S . -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DCMAKE_PREFIX_PATH="$torch_prefix"
!cmake --build build -j2

Torch CMake prefix: /usr/local/lib/python3.12/dist-packages/torch/share/cmake
-- The CXX compiler identification is GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found CUDA: /usr/local/cuda (found version "12.8")
-- The CUDA compiler identification is NVIDIA 12.8.93 with host compiler GNU 11.4.0
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
-- Found CUDAToolkit: /usr/local/cuda/include (found version "12.8.93")
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- PyTorch: CUDA detected: 12.8
-- PyTorch: CUDA nvcc is: /usr/local/cuda/bin/nvcc
-- PyTorch: CUDA toolkit

In [7]:
!./build/Change --data_dir /kaggle/working/data --output_dir /kaggle/working/output --generate_only
!python3 scripts/train_yolo_then_unet.py --dataset_dir /kaggle/working/output/digit_dataset --output_dir /kaggle/working/output

SECTION 0 -- DEPENDENCIES
Using LibTorch + OpenCV pipeline
SECTION 1 -- CONFIGURATION
Device=cuda | gpus=2 | detector_device=cuda:0 | unet_device=cuda:1 | parallel_training=on | detector_batch=256 | unet_batch=256
SECTION 2 -- DATA GENERATION
Loading raw MNIST IDX files...
MNIST loaded. Train=60000 Test=10000
Generating split 'train' with 100000 scenes
  train: 2000/100000
  train: 4000/100000
  train: 6000/100000
  train: 8000/100000
  train: 10000/100000
  train: 12000/100000
  train: 14000/100000
  train: 16000/100000
  train: 18000/100000
  train: 20000/100000
  train: 22000/100000
  train: 24000/100000
  train: 26000/100000
  train: 28000/100000
  train: 30000/100000
  train: 32000/100000
  train: 34000/100000
  train: 36000/100000
  train: 38000/100000
  train: 40000/100000
  train: 42000/100000
  train: 44000/100000
  train: 46000/100000
  train: 48000/100000
  train: 50000/100000
  train: 52000/100000
  train: 54000/100000
  train: 56000/100000
  train: 58000/100000
  train: 60

In [8]:
!ls -lah /kaggle/working/output
!find /kaggle/working/output -maxdepth 3 -type f | head -n 80
!sed -n '1,160p' /kaggle/working/output/report.txt

total 30M
drwxr-xr-x 4 root root 4.0K Apr  1 22:09 .
drwxr-xr-x 5 root root 4.0K Apr  1 13:34 ..
-rw-r--r-- 1 root root  153 Apr  1 13:37 data_yolo.yaml
drwxr-xr-x 5 root root 4.0K Apr  1 13:37 digit_dataset
-rw-r--r-- 1 root root 200K Apr  1 13:37 mnistdd_rgb_samples.png
-rw-r--r-- 1 root root  496 Apr  1 22:09 pipeline_summary.json
-rw-r--r-- 1 root root 1.2K Apr  1 22:09 report.txt
-rw-r--r-- 1 root root  30M Apr  1 20:09 unet_best.pth
-rw-r--r-- 1 root root 1020 Apr  1 22:03 unet_history.csv
-rw-r--r-- 1 root root  134 Apr  1 15:35 yolo_test_metrics.json
drwxr-xr-x 3 root root 4.0K Apr  1 15:34 yolo_train
/kaggle/working/output/digit_dataset/data.yaml
/kaggle/working/output/digit_dataset/labels/val.cache
/kaggle/working/output/digit_dataset/labels/train.cache
/kaggle/working/output/digit_dataset/labels/test.cache
/kaggle/working/output/mnistdd_rgb_samples.png
/kaggle/working/output/yolo_test_metrics.json
/kaggle/working/output/pipeline_summary.json
/kaggle/working/output/report.txt